In [21]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


# 0. Read the data

In [22]:
from config import PER_ORGANISM_PER_CLASS_ALLELES
DATA_PATH = "/home/denklewer/Data/schreiber/"
# read total data

total_df =  pd.read_csv("{}schreiber_total_data.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(total_df['allele']))

per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    per_allele_mt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    per_allele_wt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";")
    
def defineOrganism(allele_name):
    for organism_to_use in PER_ORGANISM_PER_CLASS_ALLELES.keys():
        per_class_alleles = PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use]
        for class_to_use, alleles in per_class_alleles.items():
            if allele_name in alleles:
                return organism_to_use
    print("organism not  found")
    return None


def defineClass(allele_name):
    organism =  defineOrganism(allele_name)
    per_class_alleles = PER_ORGANISM_PER_CLASS_ALLELES[organism]
    for class_to_use, alleles in per_class_alleles.items():
        if allele_name in alleles:
            return class_to_use
    print("not found")
    return None

    
ALLELES

['H-2IEd', 'H-2Ld']

In [23]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

In [24]:
METHOD = "mhcnuggets"

## 0.1. make result dicts

In [25]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

## 0.2 Read decoy data

In [26]:


per_allele_decoy = dict()
for allele_name in ALLELES:
    organism =  defineOrganism(allele_name).lower()
    class_to_use = defineClass(allele_name)
    DECOY_DATA_PATH = f"/home/denklewer/Data/decoy/sampled_decoy_data_{organism}_class_{class_to_use}.csv"
    decoy_df =  pd.read_csv(DECOY_DATA_PATH)
    decoy_df['length'] = decoy_df['peptide'].str.len()
    
    per_allele_decoy[allele_name] = decoy_df.copy()

# 1. Predict with mhcnuggets

In [27]:
# importing the predict module
from mhcnuggets.src.predict import predict
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import numpy as np
import pandas as pd

In [28]:
            
def filterByLength(per_allele_df, length_column='length'):
    #Filter for mhcnuggets <=15 length for class 1 and <=30 for class 2
    for allele_name in per_allele_df.keys():
        
        initial_len = len(per_allele_df[allele_name])
        if defineClass(allele_name) == 'I':
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 15]
        else:
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 30]
        result_len = len(per_allele_df[allele_name])
        print(f"Filtered out {initial_len - result_len} samples")
        

    
    

In [29]:
%%time
print("decoy ")
filterByLength(per_allele_decoy)

print(f"\t mt")
filterByLength(per_allele_mt_df)
print(f"\t wt")
filterByLength(per_allele_wt_df)

decoy 
Filtered out 0 samples
Filtered out 0 samples
	 mt
Filtered out 0 samples
Filtered out 0 samples
	 wt
Filtered out 0 samples
Filtered out 0 samples
CPU times: user 32 ms, sys: 277 µs, total: 32.3 ms
Wall time: 28.6 ms


In [30]:
# different allele naming.
TARGET_ALLELES = dict()

## HARDCODE
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    elif allele_name == 'HLA_DRB1_01_01':
        target_allele_name = 'HLA-DRB101:01'
    elif allele_name == 'HLA_DRB1_04_01':
        target_allele_name = 'HLA-DRB104:01'
    elif allele_name == 'HLA_DRB1_04_05':
        target_allele_name = 'HLA-DRB104:05'    
    else:
        target_allele_name = normalize_allele_name(allele_name).replace("*", "")
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'H-2IEd': 'H-2-IEd', 'H-2Ld': 'H-2-Ld'}

In [31]:
# predicting new line separated peptides present in the peptides_path file 
# for MHC class_I allele HLA-A02:01. Since a MS_BA blend model is available,
# this prediction will use the blended model (HLA-A02:01_BA_to_HLAp.h5)
def predict_with_mhcnuggets(per_allele_df, per_allele_scores_for_pecentiles = None):
    # save to temp dir
    TEMPDIR = '../TMP/'
    for allele_name in ALLELES:
        df =  per_allele_df[allele_name].copy()
        df = df[df['peptide'].notnull()]['peptide'] # remove nans
        df.to_csv(f"{TEMPDIR}mchnuggets_data_{allele_name}", index=False, header=False)    
    # predict
    for allele_name in ALLELES:
        class_to_use = defineClass(allele_name)
        target_allele = TARGET_ALLELES[allele_name]
        print(f"---{target_allele}")
        predict(class_=f'{class_to_use}',
                peptides_path=f'{TEMPDIR}mchnuggets_data_{allele_name}', 
                mhc=target_allele,
                output=f'{TEMPDIR}mchnuggets_predictions_{allele_name}')
    # read predictions from temp
    per_allele_predictions= {
        allele_name :pd.read_csv(f"{TEMPDIR}mchnuggets_predictions_{allele_name}").rename(columns={'ic50':'affinity'}) for allele_name in ALLELES
    }
    
    
    for allele_name in ALLELES:
        source_df = per_allele_df[allele_name].copy()
        source_df = source_df[source_df['peptide'].notnull()]['peptide'] # remove nans
        assert len(source_df) == len(per_allele_predictions[allele_name])
        
        
    for allele_name in per_allele_predictions.keys(): 
        df_result = per_allele_predictions[allele_name]
        # add binder label
        per_allele_predictions[allele_name] = per_allele_df[allele_name].merge(df_result, on=['peptide'], how='left')
        # log-scale affinity
        
        per_allele_predictions[allele_name]['score'] = 1 - np.log(per_allele_predictions[allele_name]['affinity'])/np.log(50000)
        
        if per_allele_scores_for_pecentiles:
            
            # get scores for percentile
            scores = per_allele_predictions[allele_name]['score']
            ref_scores = per_allele_scores_for_pecentiles[allele_name].copy()
            ref_scores = ref_scores.sort_values('score')['score']
            pct_increment = 1./len(ref_scores)
            per_allele_predictions[allele_name]['percentile_rank'] = (0.5 + 0.5*ref_scores.searchsorted(scores, side='left') + 0.5*ref_scores.searchsorted(scores, side='right'))*pct_increment
            per_allele_predictions[allele_name]['percentile_rank'] = 100 - per_allele_predictions[allele_name]['percentile_rank']*100
            
    
    
    return per_allele_predictions
    

## create tempfiles for mhcnuggets



## First calculate scores for decoy

In [32]:
per_allele_decoy_predictions = predict_with_mhcnuggets(per_allele_decoy)

---H-2-IEd
Predicting for 200000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-IEd
No BA_to_HLAp model found, predicting with BA model...
---H-2-Ld
Predicting for 200000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-Ld
No BA_to_HLAp model found, predicting with BA model...


In [33]:
%%time
print(f"--mt")
per_allele_mt_predictions = predict_with_mhcnuggets(per_allele_mt_df, per_allele_decoy_predictions)
per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions  

print(f"--wt")
per_allele_wt_predictions = predict_with_mhcnuggets(per_allele_wt_df, per_allele_decoy_predictions)
per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions  


--mt
---H-2-IEd
Predicting for 134 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-IEd
No BA_to_HLAp model found, predicting with BA model...
---H-2-Ld
Predicting for 51 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-Ld
No BA_to_HLAp model found, predicting with BA model...
--wt
---H-2-IEd
Predicting for 134 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-IEd
No BA_to_HLAp model found, predicting with BA model...
---H-2-Ld
Predicting for 51 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-Ld
No BA_to_HLAp model found, predicting with BA model...
CPU times: user 1.98 s, sys: 190 ms, total: 2.17 s
Wall time: 1.82 s


In [34]:
per_method_per_allele_wt_predictions[METHOD][ALLELES[0]]

,peptide,length,allele,source_sequence_name,affinity,score,percentile_rank
0,NTGRGYVLI,9,H-2IEd,seq0,17486.66,0.097099,39.39175
1,TGRGYVLIR,9,H-2IEd,seq1,18872.05,0.090052,42.27925
2,GRGYVLIRI,9,H-2IEd,seq2,9229.54,0.156160,22.06475
3,RGYVLIRIL,9,H-2IEd,seq3,7935.15,0.170126,19.13775
4,GYVLIRILR,9,H-2IEd,seq4,4496.11,0.222630,11.27675
...,...,...,...,...,...,...,...
129,RPDNTGRGYVLIRILRRAVRYSH,23,H-2IEd,seq129,4301.73,0.226715,10.81925
130,PDNTGRGYVLIRILRRAVRYSHE,23,H-2IEd,seq130,4504.15,0.222465,11.29675
131,GRPDNTGRGYVLIRILRRAVRYSH,24,H-2IEd,seq131,5317.72,0.207119,13.19625
132,RPDNTGRGYVLIRILRRAVRYSHE,24,H-2IEd,seq132,3861.87,0.236684,9.76025


# 2. Join predictions

In [35]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':['source_sequence_name', 'score', 'affinity', 'percentile_rank']

}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df


def getPredictionResult(total_results, allele_name, method):
    df = total_results[method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df
    

# def getPredictionResult(total_results, region, allele_name, method):
#     df = total_results[region][method][allele_name][per_method_columns[method]].copy()
#     df = rename_columns(df, method)
#     return df


In [36]:
def joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions):
    per_allele_result = dict()
    methods = list(per_method_per_allele_mt_predictions.keys())
    for allele_name in ALLELES:
        print(f"allele {allele_name}")
        df_source_wt = per_allele_wt_df[allele_name].copy()
        df_source_mt = per_allele_mt_df[allele_name].copy()

        for method in methods:
            print(f"method {method}")
            df_method_wt = getPredictionResult(per_method_per_allele_wt_predictions, allele_name, method)
            df_method_mt = getPredictionResult(per_method_per_allele_mt_predictions, allele_name, method)
            df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
            df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
        leave_columns = [ 'source_sequence_name', 'allele', 'length']
        df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
        df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
        # rename id back 


        df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
        per_allele_result[allele_name] = df_result
    return per_allele_result
                
                

              

In [37]:
per_allele_result = joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions)

allele H-2IEd
method mhcnuggets
allele H-2Ld
method mhcnuggets


# 5. For now just write to files

In [38]:
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [39]:
# for decoy_key in DECOY_PREFIXES:

for allele_name in ALLELES:
    total_df = per_allele_result[allele_name]
    allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}{}_predictions_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,allele_name),sep=";", index=False)

In [40]:
total_df

,peptide_mt,length,allele,source_sequence_name,mhcnuggets_score_mt,mhcnuggets_affinity_mt,mhcnuggets_percentile_rank_mt,peptide_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,mhcnuggets_percentile_rank_wt
0,NCLPSLPF,8,H-2Ld,seq0,0.110630,15105.06,13.80725,NCLPSLPF,0.110630,15105.06,13.80725
1,CLPSLPFI,8,H-2Ld,seq1,0.268632,2733.23,3.12725,CLPSLPFI,0.268632,2733.23,3.12725
2,LPSLPFII,8,H-2Ld,seq2,0.322616,1524.07,2.03425,LPSLPFII,0.322616,1524.07,2.03425
3,PSLPFIIP,8,H-2Ld,seq3,0.035647,33998.91,64.79675,PSLPFIIP,0.035647,33998.91,64.79675
4,SLPFIIPW,8,H-2Ld,seq4,0.029858,36196.46,77.28825,SLPFIIPW,0.029858,36196.46,77.28825
5,LPFIIPWG,8,H-2Ld,seq5,0.122320,13310.41,11.89925,LPFIIPWG,0.122320,13310.41,11.89925
6,PFIIPWGL,8,H-2Ld,seq6,0.235020,3932.04,4.05975,PFIIPWGL,0.235020,3932.04,4.05975
7,FIIPWGLI,8,H-2Ld,seq7,0.059988,26127.02,33.16775,FIIPWGLI,0.059988,26127.02,33.16775
8,ANCLPSLPF,9,H-2Ld,seq8,0.131410,12063.58,10.67425,ANCLPSLPF,0.131410,12063.58,10.67425
9,NCLPSLPFI,9,H-2Ld,seq9,0.285132,2286.36,2.73725,NCLPSLPFI,0.285132,2286.36,2.73725
